In [0]:
from pyspark.sql.functions import current_timestamp, lit

def ingest_csv_to_bronze(file_name, table_name):
    """
    Reusable function to ingest any raw CSV file into a Bronze Delta table
    with audit metadata columns.
    """
    # 1. Dynamically build the file path based on the file name passed in
    file_path = f"file:/Workspace/Repos/Git/education-analytics-lakehouse/data/sample/{file_name}"
    
    print(f"Starting ingestion for: {file_name}...")
    
    # 2. Read the raw CSV file
    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )
    
    # 3. Add audit metadata columns for lineage
    bronze_df = (
        raw_df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(file_path))
    )
    
    # 4. Save as a managed Delta table
    (
        bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    
    print(f"Successfully saved to Bronze table: {table_name} ({bronze_df.count()} records)\n")

# ==========================================
# Run the function for multiple files at once!
# ==========================================

# List of tuples: (csv_file_name, target_bronze_table_name)
datasets_to_ingest = [
    ("universities.csv", "bronze_universities"),
    ("courses.csv", "bronze_courses"),
    ("applications.csv", "bronze_applications")
]

# Loop through the list and process each file automatically
for csv_file, table_name in datasets_to_ingest:
    ingest_csv_to_bronze(csv_file, table_name)